# SWaT Anomaly IDS Poisoning — Narval Interactive Notebook

Layer-1 notebook for the paper run on Narval. Use this for:

* inspection and dry-run on a login node or a short interactive `salloc`
* regenerating tables and figures after the Slurm arrays finish
* the LSTM-AE score-distribution diagnostic (cell 5.1b in the original notebook)

Heavy execution (the 468-run grid) **does not happen here**. That lives in
`slurm/10_clean_baselines.sh` + `slurm/2?_attack_*.sh`. This notebook imports
from `src/` and reads checkpoints written by those jobs.

### Before running

1. Activate the venv: `source venv/bin/activate`
2. Export paths if you haven't:
   ```
   export SWAT_DATA_DIR=$SCRATCH/swat_data
   export SWAT_OUTPUT_DIR=$SCRATCH/swat_paper_run
   ```
3. Confirm `$SWAT_DATA_DIR/{normal.csv,attack.csv}` exist.


## 1. Environment check

In [ ]:
import os, sys
from pathlib import Path

# Make the package importable when running the notebook from the project root
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from src.config import CONFIG, TUNED_PARAMS, get_data_paths, get_output_dir

print("Project root :", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
print("SWAT_DATA_DIR  :", os.environ.get("SWAT_DATA_DIR", "(unset — will default to $SCRATCH/swat_data)"))
print("SWAT_OUTPUT_DIR:", os.environ.get("SWAT_OUTPUT_DIR", "(unset — will default to $SCRATCH/swat_paper_run)"))

DATA_DIR, NORMAL_CSV, ATTACK_CSV = get_data_paths()
OUTPUT_DIR = get_output_dir()
print("DATA_DIR     :", DATA_DIR)
print("NORMAL_CSV   :", NORMAL_CSV)
print("ATTACK_CSV   :", ATTACK_CSV)
print("OUTPUT_DIR   :", OUTPUT_DIR)


## 2. Load & preprocess — shape check only (no training)

In [ ]:
from src.data import load_raw, preprocess_swat, create_splits, create_lstm_ae_splits

df_n_raw, df_a_raw = load_raw(NORMAL_CSV, ATTACK_CSV)
print(f"normal.csv: {len(df_n_raw):>8} rows, {df_n_raw.shape[1]} cols")
print(f"attack.csv: {len(df_a_raw):>8} rows, {df_a_raw.shape[1]} cols")

df, FEATURE_COLS, df_n_clean, df_a_clean = preprocess_swat(df_n_raw, df_a_raw)
print(f"\nCombined: {len(df):,} samples × {len(FEATURE_COLS)} features")
print(f"Attack ratio: {df['label'].mean():.4f}")
print(f"LSTM-AE blocks: normal={len(df_n_clean):,}  attack={len(df_a_clean):,}")


In [ ]:
# Pointwise split at seed=42 — shapes only
X_tr, X_n, X_v, X_te, y_tr, y_v, y_te, _ = create_splits(
    df, FEATURE_COLS, CONFIG["TEST_SIZE"], CONFIG["VAL_SIZE"], seed=42)
print(f"[pointwise seed=42] train={len(X_tr):,} (normal={len(X_n):,})  "
      f"val={len(X_v):,}  test={len(X_te):,}  attack ratio={y_te.mean():.4f}")

# LSTM-AE contiguous split at seed=42 — shapes only
X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm, X_sq_te, y_sq_te, _ = create_lstm_ae_splits(
    df_n_clean, df_a_clean, FEATURE_COLS, seed=42)
print(f"[lstm-ae seed=42]  train={len(X_sq_tr):,}  val-norm={len(X_sq_vn):,}  "
      f"val-mixed={len(X_sq_vm):,}  test={len(X_sq_te):,}  attack ratio={y_sq_te.mean():.4f}")


## 3. One-combo dry run (optional)

Runs **one** clean baseline for a single (model, seed) to confirm the pipeline
lights up end-to-end. Pick a cheap model (`iforest` or `histogram`) so this
stays under a minute.


In [ ]:
from src.run_clean import run_one_clean

# Small model, single seed — ~10-30 s
result = run_one_clean("iforest", seed=42)
print("F1={f1:.4f}  FNR={fnr:.4f}  t={time:.1f}s".format(**result))


## 4. LSTM-AE score-distribution diagnostic

Mirrors cell 5.1b of the original notebook. Trains one LSTM-AE at seed=42,
prints AUC / AP / separation gap, and saves the score histogram to
`$SWAT_OUTPUT_DIR/figures/lstm_ae_score_distribution.png`.

> This is a ~5-10 min run on a full A100 and writes a real figure. Run only
> after you have a GPU (interactive `salloc --gres=gpu:a100:1 --time=0:30:00`).


In [ ]:
import numpy as np, matplotlib.pyplot as plt, gc, torch
from sklearn.metrics import roc_auc_score, average_precision_score

from src.eval_utils import set_seed
from src.models import create_model

seed = 42
set_seed(seed)

# Reuse the seed=42 LSTM-AE split computed above
det = create_model("lstm_ae", len(FEATURE_COLS), seed=seed)
det.train(X_sq_tr, X_sq_vn, X_sq_vm, y_sq_vm)

scores = det.decision_scores(X_sq_te)
y_win  = det._create_sequence_labels(y_sq_te)
n = min(len(scores), len(y_win))
scores, y_win = scores[:n], y_win[:n]

n_norm = int((y_win == 0).sum()); n_atk = int((y_win == 1).sum())
s_norm, s_atk = scores[y_win == 0], scores[y_win == 1]
gap = (s_atk.min() - s_norm.max()) if (n_atk and n_norm) else float("nan")
auc = roc_auc_score(y_win, scores)
ap  = average_precision_score(y_win, scores)

print(f"Total test windows : {n:,}")
print(f"  Normal           : {n_norm:,}")
print(f"  Attack           : {n_atk:,}  ({y_win.mean():.4f})")
print(f"Threshold (F1-opt) : {det.threshold:.6e}")
print(f"Normal rec-err     : min={s_norm.min():.3e}  max={s_norm.max():.3e}  mean={s_norm.mean():.3e}")
print(f"Attack rec-err     : min={s_atk.min():.3e}  max={s_atk.max():.3e}  mean={s_atk.mean():.3e}")
print(f"Gap (min-atk - max-norm): {gap:.3e}   ({'SEPARABLE' if gap > 0 else 'OVERLAPPING'})")
print(f"AUC                : {auc:.6f}")
print(f"Average precision  : {ap:.6f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].hist(s_norm, bins=60, alpha=0.7, label=f"Normal (n={n_norm:,})", color="#4C72B0")
ax[0].hist(s_atk,  bins=60, alpha=0.7, label=f"Attack (n={n_atk:,})", color="#C44E52")
ax[0].axvline(det.threshold, color="black", linestyle="--",
              label=f"threshold={det.threshold:.2e}")
ax[0].set(xlabel="Reconstruction error", ylabel="Window count",
          title="LSTM-AE score distribution (linear)")
ax[0].legend(loc="upper right"); ax[0].grid(alpha=0.3)

ax[1].hist(np.log10(s_norm + 1e-12), bins=60, alpha=0.7, label="Normal", color="#4C72B0")
ax[1].hist(np.log10(s_atk  + 1e-12), bins=60, alpha=0.7, label="Attack", color="#C44E52")
ax[1].axvline(np.log10(det.threshold + 1e-12), color="black", linestyle="--", label="threshold")
ax[1].set(xlabel="log10(reconstruction error)", ylabel="Window count",
          title="LSTM-AE score distribution (log scale)")
ax[1].legend(loc="upper right"); ax[1].grid(alpha=0.3)

plt.suptitle(f"LSTM-AE clean-baseline diagnostic — AUC={auc:.4f}, AP={ap:.4f}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
out_png = OUTPUT_DIR / "figures" / "lstm_ae_score_distribution.png"
out_png.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_png, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out_png}")

del det
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()


## 5. Inspect Slurm checkpoints (partial-progress view)

Shows which combos are already done and which are still missing. Safe to call
mid-grid — reads JSON-per-combo files directly.


In [ ]:
from src.aggregate import load_clean, load_attacks, report_missing

clean_df    = load_clean(OUTPUT_DIR)
poisoned_df = load_attacks(OUTPUT_DIR)
print(f"Clean checkpoints    : {len(clean_df)} / {len(CONFIG['MODELS']) * len(CONFIG['SEEDS'])}")
print(f"Poisoned checkpoints : {len(poisoned_df)} / "
      f"{len(CONFIG['MODELS']) * len(CONFIG['SEEDS']) * len(CONFIG['ATTACKS']) * len(CONFIG['POISON_RATES'])}")

report = report_missing(OUTPUT_DIR)
if report["clean_missing"]:
    print("\nStill missing (clean, model seed):")
    for m, s in report["clean_missing"]:
        print(f"  {m} {s}")
if report["attack_missing"]:
    print(f"\nStill missing (attack) — {len(report['attack_missing'])} combos")
    for a, m, s, r in report["attack_missing"][:20]:
        print(f"  {a} {m} s={s} r={r}")
    if len(report['attack_missing']) > 20:
        print(f"  ... and {len(report['attack_missing']) - 20} more")


## 6. Build tables and figures from whatever is done

Calls `src.aggregate.main()` which writes:
* `checkpoints/{clean_baselines,attack_checkpoint}.csv`
* `all_results.csv`, `table_T4_*.csv`, `table_T5_*.csv`
* `compute_cost.csv`, `multi_criteria_ranking.csv`
* `figures/F3_*.png` … `figures/F8_*.png`
* `run_summary.txt`


In [ ]:
from src.aggregate import main as aggregate_main
aggregate_main()


## 7. Table T4 preview (clean baselines)

In [ ]:
import pandas as pd
T4_path = OUTPUT_DIR / "table_T4_clean_baselines.csv"
if T4_path.exists():
    T4 = pd.read_csv(T4_path)
    display(T4.round(4))
else:
    print("T4 not yet available — run the clean-baseline array first.")


## 8. Table T5 preview (poisoning impact)

In [ ]:
T5_path = OUTPUT_DIR / "table_T5_poisoning_impact.csv"
if T5_path.exists():
    T5 = pd.read_csv(T5_path)
    display(T5.round(4))
else:
    print("T5 not yet available — run the attack arrays first.")


## 9. Multi-criteria ranking preview

In [ ]:
mcr_path = OUTPUT_DIR / "multi_criteria_ranking.csv"
if mcr_path.exists():
    display(pd.read_csv(mcr_path).round(4))
else:
    print("Multi-criteria ranking not yet available.")


## 10. Done

When `run_summary.txt` reports 0 missing combos in both clean and attack sets,
the paper run is complete. Copy `$SWAT_OUTPUT_DIR/figures/` and the
`table_T*.csv` files off Narval for the paper.
